# Push-T in-context learning transformer (v1)

**Question:** can a frozen transformer infer an unfamiliar Push-T goal from demonstrations and push the T there, without ever
seeing the goal's coordinates and without weight updates? (`CODEX_CONTEXT.md`.) See `state_bc/README.md` for the log.

**What it is trained on.** Prompts built from demonstrations by the two frozen full-state teachers in `PushTVarGoal-v1`:
Jimmy's goal-conditioned PPO and the BC teacher distilled from it (run 4), half the prompts each. One prompt = one teacher and
one goal: `k` demonstration episodes of that teacher reaching that goal from different starts (each subsampled every 4th step,
ending at the final step), then a state from **another** episode of the same teacher and goal. The target is the teacher's action at that
state. Every state the transformer sees has the goal fields (indices 21:24) removed; the demonstrations are its only clue.
Training goals are in Q1–Q3 only; Q4 goals are held out for the final test.

| Split | Goals | Use |
|---|---|---|
| `train` | 2,048 in Q1–Q3 | training prompts |
| `val` | 128 in Q1–Q3 (unseen) | checkpoint selection |
| `test_in` | 128 in Q1–Q3 (unseen) | final test, in distribution |
| `test_q4` | 128 in Q4 | final test, held-out quadrant |

10 noise-free demonstrations per goal and teacher. Context sizes `k` ∈ {0, 1, 2, 4, 8} episodes (up to 200 state-action pairs,
401 tokens). Model: GPT-2, 12 layers, width 256, 8 heads (9.75M parameters), query-action MSE, AdamW 1e-4, batch 64.

**Steps:** 1–4 setup → 5 demonstrations (cached on Drive, about 10 minutes the first time) → 6 **pilot** (3,000 steps, then checks that
correct-goal demonstrations beat wrong-goal ones) → 7 full training (resumes automatically if Colab disconnects: just rerun
the notebook) → 8 final evaluation.

Select **Runtime → Change runtime type → L4 GPU**. Upload only this notebook.

In [ ]:
from pathlib import Path
import os, sys, json, re, subprocess, datetime, shlex, shutil, base64, hashlib

MANISKILL_COMMIT = "a4a4f9272ad64b1564035874b605ceb687b63ed8"  # v3.0.1, same as the teacher notebooks
MS_ROOT = Path("/content/ManiSkill")
WORK_ROOT = Path("/content/pusht_state_bc")
WORK_ROOT.mkdir(parents=True, exist_ok=True)
DRIVE_BASE = Path("/content/drive/MyDrive/182_final_project/state_bc")
RUN_LABEL = "icl_transformer_v1"
JIMMY_COMMIT = "c87d8ae3669c10894d765df2d0dd4b61c010e272"

# Frozen teachers (run 4 session on Drive)
TEACHER_DIR = DRIVE_BASE / "20261123T230947Z_run4_bc_from_jimmy" / "teachers"
PPO_TEACHER = TEACHER_DIR / "ppo_expert.pt"    # Jimmy's ppo_vargoal_seed1_50M.pt
BC_TEACHER = TEACHER_DIR / "bc_teacher.pt"     # distilled BC, run 4

# Demonstrations: cached on Drive and reused by later sessions
DATA_TAG = "icl_data_v1"
SPLITS = ["train", "val", "test_in", "test_q4"]
GOALS = [2048, 128, 128, 128]
EPISODES_PER_GOAL = 10

# Transformer
KS = [0, 1, 2, 4, 8]          # demonstration episodes per prompt
STRIDE = 4                    # keep every 4th step of a demonstration (25 pairs per episode)
BATCH_SIZE = 64
LR = 1e-4
PILOT_ITERS = 3_000
ITERS = 100_000
EVAL_EVERY = 2_500
SEED = 0
TRAIN_DIR = DRIVE_BASE / "icl_runs" / f"transformer_v1_seed{SEED}"   # fixed path, so reruns resume

# Final closed-loop evaluation
EVAL_GOALS = 128
EVAL_STARTS = 4               # new starts per goal (512 episodes per condition)

SESSION = datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + RUN_LABEL
subprocess.run(["nvidia-smi"], check=True)

## 1. Drive and logging

This session's logs and evaluations go to a new folder; demonstrations and the training run use fixed folders so they are reused.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
OUTPUT_ROOT = DRIVE_BASE / SESSION
OUTPUT_ROOT.mkdir(parents=True, exist_ok=False)
LOG = OUTPUT_ROOT / "notebook.log"
for p in (PPO_TEACHER, BC_TEACHER):
    assert p.exists(), f"Missing teacher {p}: check TEACHER_DIR"
print("Session folder:", OUTPUT_ROOT)


def run(command, show=None, **kwargs):
    """Run a command and append all output to notebook.log. Prints every line, or only lines
    matching the regex `show`. On failure, raises with the last 40 lines."""
    command = list(map(str, command))
    print("$", shlex.join(command), flush=True)
    pattern = re.compile(show) if show else None
    tail = []
    with LOG.open("a") as log:
        log.write("\n$ " + shlex.join(command) + "\n")
        with subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                              text=True, bufsize=1, **kwargs) as process:
            try:
                for line in process.stdout:
                    if pattern is None or pattern.search(line):
                        print(line, end="", flush=True)
                    log.write(line)
                    tail = (tail + [line.rstrip()])[-40:]
                code = process.wait()
            except BaseException:
                process.kill()
                raise
    if code:
        raise RuntimeError(f"Command failed (exit {code}). Log: {LOG}\n" + "\n".join(tail))

## 2. Install ManiSkill v3.0.1 under Python 3.11, plus Hugging Face Transformers 4.46.3

In [ ]:
run(["apt-get", "update", "-qq"])
run(["apt-get", "install", "-y", "-qq", "libvulkan1", "libvulkan-dev", "vulkan-tools", "libglvnd-dev"])
if not (MS_ROOT / ".git").exists():
    run(["git", "clone", "https://github.com/mani-skill/ManiSkill.git", MS_ROOT])
run(["git", "-C", MS_ROOT, "checkout", "--detach", MANISKILL_COMMIT])
assert subprocess.check_output(["git", "-C", str(MS_ROOT), "rev-parse", "HEAD"], text=True).strip() == MANISKILL_COMMIT

run([sys.executable, "-m", "pip", "install", "-q", "uv==0.9.11"])
TRAIN_ENV = WORK_ROOT / "python311"
TRAIN_PY = TRAIN_ENV / "bin/python"
if not TRAIN_PY.exists():
    run([sys.executable, "-m", "uv", "venv", "--python", "3.11", "--seed", TRAIN_ENV])
constraints = WORK_ROOT / "constraints.txt"
constraints.write_text("numpy==1.26.4\nopencv-python==4.11.0.86\nsapien==3.0.3\ntorch==2.6.0\n")
run([TRAIN_PY, "-m", "pip", "install", "-q", "torch==2.6.0", "--index-url", "https://download.pytorch.org/whl/cu124"])
run([TRAIN_PY, "-m", "pip", "install", "-q", "--constraint", constraints, "-e", MS_ROOT,
     "numpy", "sapien", "h5py", "tyro", "imageio[ffmpeg]", "matplotlib", "transformers==4.46.3"])

for filename, library, vulkan in [
    ("/usr/share/vulkan/icd.d/nvidia_icd.json", "libGLX_nvidia.so.0", True),
    ("/usr/share/glvnd/egl_vendor.d/10_nvidia.json", "libEGL_nvidia.so.0", False),
]:
    path = Path(filename)
    if not path.exists():
        path.parent.mkdir(parents=True, exist_ok=True)
        icd = {"library_path": library}
        if vulkan:
            icd["api_version"] = "1.2.155"
        path.write_text(json.dumps({"file_format_version": "1.0.0", "ICD": icd}))
os.environ["VK_ICD_FILENAMES"] = "/usr/share/vulkan/icd.d/nvidia_icd.json"
os.environ["MS_SKIP_ASSET_DOWNLOAD_PROMPT"] = "1"
os.environ["PYTHONUNBUFFERED"] = "1"
run([TRAIN_PY, "-c", "import torch, sapien, mani_skill, transformers; print('torch', torch.__version__, 'mani_skill', "
     "mani_skill.__version__, 'transformers', transformers.__version__); assert torch.cuda.is_available(); "
     "print(torch.cuda.get_device_name(0))"])
(OUTPUT_ROOT / "pip-freeze.txt").write_text(subprocess.check_output([str(TRAIN_PY), "-m", "pip", "freeze"], text=True))

## 3. Write the modules

`pusht_icl.py` (transformer, prompts, training, evaluation) and `pusht_state_bc.py` (environment helpers, teacher loaders)
are this repo's `state_bc/` files. `pusht_vargoal.py` and its arm-pose library are copied unchanged from Jimmy's branch at
`c87d8ae`, checked by SHA-256.

In [ ]:
%%writefile /content/pusht_state_bc/pusht_icl.py
"""In-context learning (ICL) transformer for variable-goal Push-T (v1 of CODEX_CONTEXT.md).

A frozen GPT-2-style transformer sees demonstrations from one teacher for one hidden goal, then
controls the robot from a new start. It never sees the goal: the goal fields (state indices 21:24)
are removed from every state it receives. Teachers (goal-conditioned, they do see the goal):
Jimmy's PPO and the distilled BC teacher, both in PushTVarGoal-v1.

Subcommands (run with the ManiSkill training interpreter, pusht_vargoal.py next to this file):

    collect   For each split's goal set, roll out each teacher (noise-free) E times per goal
              from different starts and save {split}_{teacher}.npz: obs [G, E, T, 31],
              actions [G, E, T, 7], first_success_step [G, E], success_at_end [G, E],
              goal_xy [G, 2], goal_quadrant [G].
    train     Train the transformer on prompts sampled on the fly from the train split;
              validation action error (correct vs wrong-goal context) on the val split.
              Saves last.pt (for resuming) and best.pt.
    offline   Teacher-forced action error per context size, correct vs wrong-goal context.
    eval      Closed-loop success with frozen weights: per split, teacher, context size and
              condition (correct-goal, wrong-goal, no context), plus the teachers themselves.

Splits (goal positions, PushTVarGoal-v1 quadrants): train Q1-Q3, val Q1-Q3, test_in Q1-Q3
(unseen goals), test_q4 Q4 (held out from transformer training). Each split's goals are an
independent seeded draw (continuous positions, so no goal repeats across splits in practice).

Prompt layout (PushT_ICL_Transformer_Spec.md): [s_1, a_1, ..., s_N, a_N, s_q] -> a_q, one token
per state or action. Support = k whole demonstration episodes of the same teacher and goal,
each subsampled every `stride` steps ending at the last step (so the final T pose is included),
N = k * ceil(T / stride). The query state comes from a different episode of the same teacher and
goal; its label is the teacher's action there.

Pure data/model code imports without ManiSkill (transformers is needed for the model).
"""

from __future__ import annotations

import argparse
import json
import math
import time
from pathlib import Path

import numpy as np
import torch
from torch import nn

import pusht_state_bc as bc

GOAL_START, GOAL_STOP = 21, 24          # extra/goal_pos in the 31-number state
TEACHERS = ("ppo", "bc")
SPLITS = {"train": (1, 2, 3), "val": (1, 2, 3), "test_in": (1, 2, 3), "test_q4": (4,)}
GOAL_SEED = {"train": 11, "val": 12, "test_in": 13, "test_q4": 14}
DEMO_SEED_BASE = {"train": 1_000_000, "val": 2_000_000, "test_in": 3_000_000, "test_q4": 4_000_000}
EVAL_SEED_BASE = {"train": 7_100_000, "val": 7_200_000, "test_in": 7_300_000, "test_q4": 7_400_000}


# --------------------------------------------------------------------------------------
# Data
# --------------------------------------------------------------------------------------

def hide_goal(x):
    """Drop the goal fields (21:24) from states [..., 31] -> [..., 28]."""
    if isinstance(x, np.ndarray):
        return np.concatenate([x[..., :GOAL_START], x[..., GOAL_STOP:]], axis=-1)
    return torch.cat([x[..., :GOAL_START], x[..., GOAL_STOP:]], dim=-1)


def snippet_steps(T, stride):
    """Time steps kept from each support episode: every `stride`-th, always including the last."""
    return torch.arange(T - 1, -1, -stride).flip(0)


def sample_goal_set(sample_fn, n, quadrants, seed):
    """Goal set for a split, reproducible from `seed` without touching the global RNG.
    sample_fn(n, quadrants) is pusht_vargoal.sample_goals (CPU)."""
    with torch.random.fork_rng(devices=[]):
        torch.manual_seed(seed)
        return sample_fn(n, quadrants)


class PromptData:
    """Teacher demonstrations grouped by goal, as tensors (kept on the GPU during training).

    obs [n_teachers, G, E, T, d_s] (goal hidden), actions [n_teachers, G, E, T, d_a],
    success [n_teachers, G, E] (reached the goal at some step).
    """

    def __init__(self, obs, actions, success, goal_xy, goal_quadrant, teachers=TEACHERS):
        self.obs, self.actions, self.success = obs, actions, success
        self.goal_xy, self.goal_quadrant, self.teachers = goal_xy, goal_quadrant, tuple(teachers)
        self.n_teachers, self.G, self.E, self.T = obs.shape[:4]

    @classmethod
    def load(cls, data_dir, split, teachers=TEACHERS, device="cpu"):
        obs, actions, success, goals = [], [], [], None
        for t in teachers:
            with np.load(Path(data_dir) / f"{split}_{t}.npz") as d:
                if goals is None:
                    goals, quadrant = d["goal_xy"], d["goal_quadrant"]
                elif not np.allclose(goals, d["goal_xy"]):
                    raise ValueError(f"{split}: teachers were collected on different goal sets")
                obs.append(hide_goal(d["obs"]))
                actions.append(d["actions"])
                success.append(d["first_success_step"] > 0)
        as_t = lambda x, dt=torch.float32: torch.as_tensor(np.stack(x), dtype=dt, device=device)  # noqa: E731
        return cls(as_t(obs), as_t(actions), as_t(success, torch.bool),
                   torch.as_tensor(goals, device=device), torch.as_tensor(quadrant, device=device), teachers)

    def episode_order(self, ti, gi, generator):
        """[B, E] episode order per (teacher, goal): successful episodes first, random within."""
        scores = torch.rand(len(ti), self.E, generator=generator, device=self.obs.device)
        return (scores + self.success[ti, gi].float()).argsort(dim=1, descending=True)

    def support(self, ti, gi, episodes, stride):
        """States and actions of the given episodes [B, k], subsampled, flattened to [B, k*P, d]."""
        B, k = episodes.shape
        steps = snippet_steps(self.T, stride).to(self.obs.device)
        idx = (ti[:, None, None], gi[:, None, None], episodes[:, :, None], steps[None, None, :])
        n = k * len(steps)
        return (self.obs[idx].reshape(B, n, self.obs.shape[-1]),
                self.actions[idx].reshape(B, n, self.actions.shape[-1]))

    def sample(self, batch, k, stride, generator, wrong_goal=False):
        """One batch of prompts with k support episodes. With wrong_goal=True the support comes from a
        different goal of the same teacher; the query (episode, step, label) is identical to the
        wrong_goal=False draw from the same generator state."""
        if k >= self.E:
            raise ValueError(f"k={k} needs more than {self.E} episodes per goal")
        dev = self.obs.device
        ti = torch.randint(self.n_teachers, (batch,), generator=generator, device=dev)
        gi = torch.randint(self.G, (batch,), generator=generator, device=dev)
        order = self.episode_order(ti, gi, generator)
        shift = torch.randint(1, max(self.G, 2), (batch,), generator=generator, device=dev)
        gw = (gi + shift) % self.G
        order_w = self.episode_order(ti, gw, generator)
        qt = torch.randint(self.T, (batch,), generator=generator, device=dev)
        qe = order[:, k]
        query_s, query_a = self.obs[ti, gi, qe, qt], self.actions[ti, gi, qe, qt]
        sg, so = (gw, order_w) if wrong_goal else (gi, order)
        support_s, support_a = self.support(ti, sg, so[:, :k], stride)
        return support_s, support_a, query_s, query_a


# --------------------------------------------------------------------------------------
# Model
# --------------------------------------------------------------------------------------

class ICLTransformer(nn.Module):
    """GPT-2 backbone (PushT_ICL_Transformer_Spec.md section 2) with numerical state/action
    projections and a linear action head read at the query token."""

    def __init__(self, d_s, d_a, state_mean, state_scale, n_layer=12, n_embd=256, n_head=8, n_positions=1024):
        super().__init__()
        from transformers import GPT2Config, GPT2Model

        self.hparams = dict(d_s=d_s, d_a=d_a, n_layer=n_layer, n_embd=n_embd, n_head=n_head, n_positions=n_positions)
        config = GPT2Config(
            vocab_size=1, n_positions=n_positions, n_embd=n_embd, n_layer=n_layer, n_head=n_head,
            n_inner=4 * n_embd, activation_function="gelu_new", resid_pdrop=0.0, embd_pdrop=0.0, attn_pdrop=0.0,
            layer_norm_epsilon=1e-5, initializer_range=0.02, scale_attn_weights=True,
            scale_attn_by_inverse_layer_idx=False, reorder_and_upcast_attn=False, add_cross_attention=False,
            use_cache=False, bos_token_id=None, eos_token_id=None)
        self.backbone = GPT2Model(config)
        self.state_encoder = nn.Linear(d_s, n_embd)
        self.action_encoder = nn.Linear(d_a, n_embd)
        self.action_head = nn.Linear(n_embd, d_a)
        self.register_buffer("state_mean", torch.as_tensor(state_mean, dtype=torch.float32))
        self.register_buffer("state_scale", torch.as_tensor(state_scale, dtype=torch.float32))

    def forward(self, support_s, support_a, query_s):
        """support_s [B, N, d_s], support_a [B, N, d_a] (N may be 0), query_s [B, d_s] -> [B, d_a]."""
        B, N = support_s.shape[:2]
        if 2 * N + 1 > self.hparams["n_positions"]:
            raise ValueError(f"prompt of {2 * N + 1} tokens exceeds {self.hparams['n_positions']}")
        q = self.state_encoder((query_s - self.state_mean) / self.state_scale)[:, None]
        if N:
            s = self.state_encoder((support_s - self.state_mean) / self.state_scale)
            a = self.action_encoder(support_a)
            q = torch.cat([torch.stack([s, a], dim=2).reshape(B, 2 * N, -1), q], dim=1)
        h = self.backbone(inputs_embeds=q).last_hidden_state[:, -1]
        return self.action_head(h)


def save_model(model, path, extra=None):
    torch.save({"hparams": model.hparams, "state_dict": model.state_dict(), **(extra or {})}, path)


def load_model(path, device="cpu"):
    ckpt = torch.load(path, map_location=device, weights_only=False)
    hp = ckpt["hparams"]
    model = ICLTransformer(state_mean=torch.zeros(hp["d_s"]), state_scale=torch.ones(hp["d_s"]), **hp)
    model.load_state_dict(ckpt["state_dict"])
    return model.to(device).eval()


class ICLActor:
    """Closed-loop controller: the frozen transformer with a fixed support per environment."""

    def __init__(self, model, support_s, support_a):
        self.model, self.support_s, self.support_a = model, support_s, support_a

    @torch.no_grad()
    def act(self, obs):
        with torch.autocast("cuda", dtype=torch.bfloat16, enabled=obs.is_cuda):
            a = self.model(self.support_s, self.support_a, hide_goal(obs.float()))
        return a.float().clamp(-1.0, 1.0)


# --------------------------------------------------------------------------------------
# Training and offline evaluation
# --------------------------------------------------------------------------------------

@torch.no_grad()
def offline_eval(model, data, ks, stride, batches=4, batch=256, seed=0):
    """Teacher-forced query-action MSE per context size: correct-goal vs wrong-goal support (same queries)."""
    model.eval()
    dev = data.obs.device
    out = {}
    for k in ks:
        row = {}
        for cond in (("none",) if k == 0 else ("correct", "wrong")):
            g = torch.Generator(device=dev).manual_seed(seed + 1000 * k)
            total = 0.0
            for _ in range(batches):
                s, a, qs, qa = data.sample(batch, k, stride, g, wrong_goal=(cond == "wrong"))
                with torch.autocast("cuda", dtype=torch.bfloat16, enabled=dev.type == "cuda"):
                    pred = model(s, a, qs)
                total += torch.nn.functional.mse_loss(pred.float().clamp(-1, 1), qa).item()
            row[cond] = total / batches
        out[str(k)] = row
    model.train()
    return out


def lr_at(it, iters, lr, warmup):
    if it < warmup:
        return lr * (it + 1) / warmup
    progress = (it - warmup) / max(1, iters - warmup)
    return lr * (0.1 + 0.9 * 0.5 * (1 + math.cos(math.pi * min(progress, 1.0))))


def train(args):
    device = torch.device(args.device)
    torch.manual_seed(args.seed)
    out = Path(args.out)
    out.mkdir(parents=True, exist_ok=True)
    ks = list(args.ks)
    train_data = PromptData.load(args.data_dir, "train", device=device)
    val_data = PromptData.load(args.data_dir, "val", device=device)
    pairs = len(snippet_steps(train_data.T, args.stride))
    print(f"train: {train_data.n_teachers} teachers x {train_data.G} goals x {train_data.E} episodes x {train_data.T} steps; "
          f"val: {val_data.G} goals. Context sizes {ks} episodes = up to {max(ks) * pairs} pairs "
          f"({2 * max(ks) * pairs + 1} tokens)", flush=True)

    flat = train_data.obs.reshape(-1, train_data.obs.shape[-1])
    pick = torch.randperm(len(flat), device=device)[:1_000_000]
    mean, _, scale = bc.compute_state_stats(flat[pick].cpu().numpy())
    model = ICLTransformer(flat.shape[-1], train_data.actions.shape[-1], mean, scale,
                           n_layer=args.n_layer, n_embd=args.n_embd, n_head=args.n_head).to(device)
    print(f"parameters: {sum(p.numel() for p in model.parameters()) / 1e6:.2f}M", flush=True)
    opt = torch.optim.AdamW(model.parameters(), lr=args.lr, weight_decay=args.weight_decay)
    gen = torch.Generator(device=device).manual_seed(args.seed)
    start, best = 0, float("inf")
    last = out / "last.pt"
    if args.resume and last.exists():
        ckpt = torch.load(last, map_location=device, weights_only=False)
        model.load_state_dict(ckpt["state_dict"])
        opt.load_state_dict(ckpt["optimizer"])
        gen.set_state(ckpt["generator"].cpu())
        start, best = ckpt["iter"], ckpt["best"]
        print(f"resumed from {last} at iteration {start}", flush=True)
    config = {**vars(args), "pairs_per_episode": pairs, "train_goals": train_data.G, "episodes_per_goal": train_data.E}
    (out / "config.json").write_text(json.dumps(config, indent=2, default=str))

    log = (out / "train_log.jsonl").open("a")
    running, n_running, t0 = 0.0, 0, time.time()
    for it in range(start, args.iters):
        for group in opt.param_groups:
            group["lr"] = lr_at(it, args.iters, args.lr, args.warmup)
        k = ks[int(torch.randint(len(ks), (1,), generator=gen, device=device).item())]
        s, a, qs, qa = train_data.sample(args.batch_size, k, args.stride, gen)
        with torch.autocast("cuda", dtype=torch.bfloat16, enabled=device.type == "cuda"):
            pred = model(s, a, qs)
        loss = torch.nn.functional.mse_loss(pred.float(), qa)
        opt.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        running += loss.item()
        n_running += 1
        done = it + 1
        if done % args.log_every == 0:
            rate = n_running / (time.time() - t0)
            print(f"iter {done} loss {running / n_running:.5f} lr {opt.param_groups[0]['lr']:.2e} "
                  f"{rate:.1f} it/s (ETA {(args.iters - done) / rate / 60:.0f} min)", flush=True)
            running, n_running, t0 = 0.0, 0, time.time()
        if done % args.eval_every == 0 or done == args.iters:
            val = offline_eval(model, val_data, ks, args.stride)
            score = float(np.mean([v.get("correct", v.get("none")) for v in val.values()]))
            print(f"iter {done} val MSE " + "  ".join(
                f"k={k}: " + "/".join(f"{c} {m:.4f}" for c, m in v.items()) for k, v in val.items()), flush=True)
            log.write(json.dumps({"iter": done, "val": val, "val_score": score}) + "\n")
            log.flush()
            if score < best:
                best = score
                save_model(model, out / "best.pt", {"iter": done, "val": val})
            torch.save({"hparams": model.hparams, "state_dict": model.state_dict(), "optimizer": opt.state_dict(),
                        "generator": gen.get_state(), "iter": done, "best": best}, last)
    save_model(model, out / "final.pt", {"iter": args.iters})
    log.close()
    print(f"Done. Best validation score {best:.5f}", flush=True)


def offline(args):
    device = torch.device(args.device)
    model = load_model(args.checkpoint, device)
    results = {}
    for split in args.splits:
        data = PromptData.load(args.data_dir, split, device=device)
        results[split] = offline_eval(model, data, args.ks, args.stride, batches=args.batches)
        print(split, json.dumps(results[split]), flush=True)
    if args.out:
        Path(args.out).write_text(json.dumps(results, indent=2))


# --------------------------------------------------------------------------------------
# Simulator: demonstration collection and closed-loop evaluation
# --------------------------------------------------------------------------------------

def load_teachers(args, env, device):
    return {"ppo": bc.load_expert(args.ppo_teacher, env, device), "bc": bc.load_policy(args.bc_teacher, device)}


def collect(args):
    """Noise-free demonstrations per split, goal and teacher; every teacher sees the same goals."""
    import pusht_vargoal as pv

    bc.configure_env(bc.SHARED_ENV_ID)
    device = torch.device(args.device)
    out = Path(args.out)
    out.mkdir(parents=True, exist_ok=True)
    env = bc.make_env(args.num_envs)
    T = bc.episode_length(env)
    try:
        teachers = load_teachers(args, env, device)
        for split, n_goals in zip(args.splits, args.goals):
            goals = sample_goal_set(pv.sample_goals, n_goals, SPLITS[split], GOAL_SEED[split]).float()
            quadrant = pv.quadrant_of(goals)
            assert set(quadrant.tolist()) <= set(SPLITS[split]), split
            goal_of_episode = torch.arange(n_goals).repeat_interleave(args.episodes)
            n = len(goal_of_episode)
            batches = math.ceil(n / args.num_envs)
            for name, policy in teachers.items():
                path = out / f"{split}_{name}.npz"
                if path.exists() and not args.overwrite:
                    print("exists, skipping:", path, flush=True)
                    continue
                obs_all, act_all, first_all, end_all = [], [], [], []
                for b in range(batches):
                    ids = goal_of_episode[b * args.num_envs:(b + 1) * args.num_envs]
                    ids = torch.cat([ids, ids[:1].expand(args.num_envs - len(ids))])  # pad the last batch
                    goal_xy = goals[ids].to(device)
                    res, obs, labels = bc.rollout(env, T, policy, seed=DEMO_SEED_BASE[split] + b, labeler=policy,
                                                  reset_options={"goal_xy": goal_xy})
                    placed = torch.as_tensor(res["goal_xy_per_episode"])
                    if not torch.allclose(placed, goal_xy.cpu(), atol=1e-4):
                        raise AssertionError("environment did not place the requested goals")
                    obs_all.append(obs)
                    act_all.append(labels)
                    first_all.append(np.asarray(res["first_success_step_per_episode"]))
                    end_all.append(np.asarray(res["success_at_end_per_episode"]))
                    print(f"{split} {name} batch {b + 1}/{batches}: success once {res['success_once']:.0%} "
                          f"at end {res['success_at_end']:.0%}", flush=True)
                shape = (n_goals, args.episodes)
                obs = np.concatenate(obs_all)[:n].reshape(*shape, T, -1)
                act = np.concatenate(act_all)[:n].reshape(*shape, T, -1)
                first = np.concatenate(first_all)[:n].reshape(shape)
                end = np.concatenate(end_all)[:n].reshape(shape)
                np.savez(path, obs=obs.astype(np.float32), actions=act.astype(np.float32), first_success_step=first,
                         success_at_end=end, goal_xy=goals.numpy(), goal_quadrant=quadrant.numpy())
                meta = dict(split=split, teacher=name, goals=n_goals, episodes_per_goal=args.episodes, steps=T,
                            env_id=bc.SHARED_ENV_ID, quadrants=list(SPLITS[split]), goal_seed=GOAL_SEED[split],
                            demo_seed_base=DEMO_SEED_BASE[split], noise_std=0.0,
                            success_once=float((first > 0).mean()), success_at_end=float(end.mean()),
                            teacher_checkpoint=str(args.ppo_teacher if name == "ppo" else args.bc_teacher))
                path.with_suffix(".json").write_text(json.dumps(meta, indent=2))
                print(json.dumps(meta), flush=True)
    finally:
        env.close()


def build_support(data, teacher, goal_ids, k, stride, seed=0):
    """Deterministic support for closed-loop evaluation: the first k episodes (successful first)
    of each listed goal for one teacher. Returns [len(goal_ids), k*P, d] states and actions."""
    dev = data.obs.device
    ti = torch.full((len(goal_ids),), data.teachers.index(teacher), device=dev)
    gi = torch.as_tensor(goal_ids, device=dev)
    order = data.episode_order(ti, gi, torch.Generator(device=dev).manual_seed(seed))
    return data.support(ti, gi, order[:, :k], stride)


def closed_loop(args):
    bc.configure_env(bc.SHARED_ENV_ID)
    device = torch.device(args.device)
    model = load_model(args.checkpoint, device)
    stride = args.stride
    results = {"checkpoint": str(args.checkpoint), "stride": stride, "starts_per_goal": args.starts, "splits": {}}
    G = args.goals
    # One simulator for all splits: ManiSkill's GPU physics is set up once per process.
    env = bc.make_env(G * args.starts)
    T = bc.episode_length(env)
    teachers = load_teachers(args, env, device)
    for split in args.splits:
        data = PromptData.load(args.data_dir, split, device=device)
        if data.G < G:
            raise ValueError(f"{split} has {data.G} goals; --goals {G} needs at least that many")
        goal_ids = torch.arange(G).repeat_interleave(args.starts)
        reset = {"goal_xy": data.goal_xy[goal_ids].float().to(device)}
        seed = EVAL_SEED_BASE[split]
        rows = {}
        for name, policy in teachers.items():
            rows[f"teacher_{name}"] = bc.rollout(env, T, policy, seed, reset_options=reset)
        wrong_ids = (goal_ids + G // 2) % G   # another goal of the same split, same teacher
        for teacher in args.context_teachers:
            for k in args.ks:
                for cond in (("none",) if k == 0 else ("correct", "wrong")):
                    ids = wrong_ids if cond == "wrong" else goal_ids
                    s, a = build_support(data, teacher, ids.tolist(), k, stride)
                    res = bc.rollout(env, T, ICLActor(model, s, a), seed, reset_options=reset)
                    rows[f"{teacher}_k{k}_{cond}"] = res
                    print(f"{split:<8} {teacher}-context k={k} {cond:<7} once {res['success_once']:.0%} "
                          f"end {res['success_at_end']:.0%}", flush=True)
        for name in teachers:
            r = rows[f"teacher_{name}"]
            print(f"{split:<8} {name} teacher (sees goal)  once {r['success_once']:.0%} end {r['success_at_end']:.0%}", flush=True)
        results["splits"][split] = {"goals": G, "episodes": G * args.starts, "seed": seed, "results": rows}
        if args.out:  # save after every split, so a disconnect keeps finished splits
            Path(args.out).write_text(json.dumps(results, indent=2))
    env.close()
    if args.out:
        Path(args.out).write_text(json.dumps(results, indent=2))
    return results


# --------------------------------------------------------------------------------------
# CLI
# --------------------------------------------------------------------------------------

def main(argv=None):
    p = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    sub = p.add_subparsers(dest="cmd", required=True)
    default_device = "cuda" if torch.cuda.is_available() else "cpu"

    teachers = argparse.ArgumentParser(add_help=False)
    teachers.add_argument("--ppo-teacher", required=True)
    teachers.add_argument("--bc-teacher", required=True)

    c = sub.add_parser("collect", parents=[teachers])
    c.add_argument("--splits", nargs="+", default=list(SPLITS))
    c.add_argument("--goals", type=int, nargs="+", default=[2048, 128, 128, 128], help="goals per split, same order")
    c.add_argument("--episodes", type=int, default=10, help="demonstrations per goal and teacher")
    c.add_argument("--num-envs", type=int, default=1024)
    c.add_argument("--out", required=True)
    c.add_argument("--overwrite", action="store_true")
    c.add_argument("--device", default=default_device)

    t = sub.add_parser("train")
    t.add_argument("--data-dir", required=True)
    t.add_argument("--out", required=True)
    t.add_argument("--ks", type=int, nargs="+", default=[0, 1, 2, 4, 8], help="support episodes per prompt, sampled per batch")
    t.add_argument("--stride", type=int, default=4, help="keep every stride-th step of a support episode")
    t.add_argument("--iters", type=int, default=100_000)
    t.add_argument("--batch-size", type=int, default=64)
    t.add_argument("--lr", type=float, default=1e-4)
    t.add_argument("--weight-decay", type=float, default=0.0)
    t.add_argument("--warmup", type=int, default=1000)
    t.add_argument("--n-layer", type=int, default=12)
    t.add_argument("--n-embd", type=int, default=256)
    t.add_argument("--n-head", type=int, default=8)
    t.add_argument("--log-every", type=int, default=250)
    t.add_argument("--eval-every", type=int, default=2500)
    t.add_argument("--seed", type=int, default=0)
    t.add_argument("--resume", action=argparse.BooleanOptionalAction, default=True)
    t.add_argument("--device", default=default_device)

    o = sub.add_parser("offline")
    o.add_argument("--checkpoint", required=True)
    o.add_argument("--data-dir", required=True)
    o.add_argument("--splits", nargs="+", default=["val", "test_in", "test_q4"])
    o.add_argument("--ks", type=int, nargs="+", default=[0, 1, 2, 4, 8])
    o.add_argument("--stride", type=int, default=4)
    o.add_argument("--batches", type=int, default=8)
    o.add_argument("--out", default="")
    o.add_argument("--device", default=default_device)

    e = sub.add_parser("eval", parents=[teachers])
    e.add_argument("--checkpoint", required=True)
    e.add_argument("--data-dir", required=True)
    e.add_argument("--splits", nargs="+", default=["test_in", "test_q4"])
    e.add_argument("--context-teachers", nargs="+", default=list(TEACHERS))
    e.add_argument("--ks", type=int, nargs="+", default=[0, 1, 2, 4, 8])
    e.add_argument("--stride", type=int, default=4)
    e.add_argument("--goals", type=int, default=128, help="goals per split (at most the collected number)")
    e.add_argument("--starts", type=int, default=4, help="closed-loop episodes per goal, from new starts")
    e.add_argument("--out", default="")
    e.add_argument("--device", default=default_device)

    args = p.parse_args(argv)
    if args.cmd == "collect" and len(args.goals) != len(args.splits):
        p.error("--goals needs one number per split")
    {"collect": collect, "train": train, "offline": offline, "eval": closed_loop}[args.cmd](args)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /content/pusht_state_bc/pusht_state_bc.py
"""Full-state behavioral cloning (BC) teachers for ManiSkill Push-T.

The default environment is PushT-VariableGoal-v1 (variable_goal_env.py): the goal T's position
changes every episode, and policies see it in the state. --env-id PushTVarGoal-v1 selects the
group's shared variable-goal environment (Jimmy's state_rl/pusht_vargoal.py, which must sit next to
this file); --env-id PushT-v1 the original fixed-goal task. --goal-quadrants limits which goal
quadrants are sampled.

Subcommands (run with the ManiSkill training interpreter):

    ppo      Run ManiSkill's state PPO baseline (examples/baselines/ppo/ppo.py) with the
             variable-goal environment registered. Extra arguments go to ppo.py.
    audit    Inspect the state observation and action spaces; write observation_contract.json.
    check-env  Verify the success test follows each episode's goal (variable-goal env).
    prepare  Convert a replayed state-mode demo file (.h5 + .json) into train/validation
             transition files, split by whole episode.
    collect  Roll out a driver policy (the expert, or a BC checkpoint for DAgger) and label
             every visited state with the expert's deterministic action.
    train    Train one MLP BC teacher on one or more transition files, evaluating closed-loop
             success on validation seeds.
    eval     Evaluate a BC checkpoint, or a PPO checkpoint (--ppo), on a seed range, with
             success broken down by goal quadrant.
    eval-quadrants  Evaluate checkpoints separately in each goal quadrant with the shared
             protocol of state_rl/pusht_state_ppo.py (quadrant q uses seed + 1000 * q, no
             reconfiguration), and check the freeze bar in every quadrant.

Transition files are .npz with obs [N, d_s], actions [N, d_a] and episode_ids [N],
plus a .json sidecar describing where they came from.

Pure data/model code is importable without ManiSkill, so it can be unit tested offline.
ManiSkill and gymnasium are imported lazily inside the simulator functions.
"""

from __future__ import annotations

import argparse
import json
import math
import time
from pathlib import Path

import numpy as np
import torch
from torch import nn

VARIABLE_GOAL_ENV_ID = "PushT-VariableGoal-v1"
SHARED_ENV_ID = "PushTVarGoal-v1"       # Jimmy's shared env, module pusht_vargoal.py (state_rl/)
GOAL_ENV_IDS = (VARIABLE_GOAL_ENV_ID, SHARED_ENV_ID)
ENV_ID = VARIABLE_GOAL_ENV_ID  # set by configure_env() from --env-id
ENV_KWARGS = {}                # set by configure_env() from --goal-quadrants / --start-quadrants
# Per-episode records exposed by the variable-goal envs and saved with collected data.
# PushTVarGoal-v1 exposes only goal_quadrant; goal_xy is then read from the goal T's pose.
EPISODE_RECORDS = ("goal_quadrant", "goal_xy", "tip_start_quadrant", "tip_start_xy", "tee_start_quadrant",
                   "tip_ik_fallback")
OBS_MODE = "state"
CONTROL_MODE = "pd_joint_delta_pos"
SIM_BACKEND = "physx_cuda"
# Features whose training std is below this are only centered, not scaled. Near-constant
# fields (for example the fixed goal position) would otherwise turn simulator noise
# into very large normalized values.
MIN_SCALED_STD = 1e-3


# --------------------------------------------------------------------------------------
# Transition files
# --------------------------------------------------------------------------------------

def save_transitions(path, obs, actions, episode_ids, meta=None):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    np.savez_compressed(path, obs=np.asarray(obs, np.float32), actions=np.asarray(actions, np.float32),
                        episode_ids=np.asarray(episode_ids, np.int64))
    meta = dict(meta or {}, transitions=int(len(obs)), episodes=int(len(np.unique(episode_ids))))
    path.with_suffix(".json").write_text(json.dumps(meta, indent=2))
    return meta


def load_transitions(paths):
    """Concatenate transition files; episode ids are made unique across files."""
    obs, actions, episode_ids, offset = [], [], [], 0
    for p in paths:
        d = np.load(p)
        obs.append(d["obs"])
        actions.append(d["actions"])
        ids = d["episode_ids"].astype(np.int64)
        episode_ids.append(ids - ids.min() + offset if len(ids) else ids)
        offset = (episode_ids[-1].max() + 1) if len(ids) else offset
    return np.concatenate(obs), np.concatenate(actions), np.concatenate(episode_ids)


# --------------------------------------------------------------------------------------
# Released demos
# --------------------------------------------------------------------------------------

def load_replayed_demos(h5_path: str | Path):
    """Return a list of episodes, each a dict with obs [T, d_s], actions [T, d_a], and metadata.

    The replayed file stores T+1 observations per episode; the last one has no action
    and is dropped, as in ManiSkill's own BC baseline.
    """
    import h5py

    h5_path = Path(h5_path)
    meta = json.loads(h5_path.with_suffix(".json").read_text())
    env_kwargs = meta["env_info"]["env_kwargs"]
    if env_kwargs.get("control_mode") not in (None, CONTROL_MODE):
        raise ValueError(f"Demo control mode {env_kwargs.get('control_mode')} != {CONTROL_MODE}")
    if env_kwargs.get("obs_mode") != OBS_MODE:
        raise ValueError(f"Demo obs mode {env_kwargs.get('obs_mode')} != {OBS_MODE}; replay with -o state")
    episodes = []
    with h5py.File(h5_path, "r") as f:
        for ep in meta["episodes"]:
            group = f[f"traj_{ep['episode_id']}"]
            obs = np.asarray(group["obs"], dtype=np.float32)
            actions = np.asarray(group["actions"], dtype=np.float32)
            if obs.ndim != 2 or actions.ndim != 2 or obs.shape[0] != actions.shape[0] + 1:
                raise ValueError(f"Unexpected shapes in traj_{ep['episode_id']}: obs {obs.shape}, actions {actions.shape}")
            success = np.asarray(group["success"]) if "success" in group else None
            episodes.append(dict(
                episode_id=int(ep["episode_id"]),
                episode_seed=ep.get("episode_seed"),
                obs=obs[:-1],
                # The simulator clips commands to the action space, so clipped labels are
                # exactly what the expert executed.
                actions=np.clip(actions, -1.0, 1.0),
                raw_actions=actions,
                success_at_end=bool(success[-1]) if success is not None else bool(ep.get("success", False)),
            ))
    return episodes, meta


def split_episodes(num_episodes: int, val_fraction: float, seed: int):
    """Split episode indices into train/validation sets, whole episodes only."""
    if not 0 < val_fraction < 1:
        raise ValueError("val_fraction must be in (0, 1)")
    rng = np.random.default_rng(seed)
    order = rng.permutation(num_episodes)
    num_val = max(1, int(round(num_episodes * val_fraction)))
    return np.sort(order[num_val:]), np.sort(order[:num_val])


def compute_state_stats(states: np.ndarray):
    mean = states.mean(axis=0)
    std = states.std(axis=0)
    scale = np.where(std < MIN_SCALED_STD, 1.0, std)
    return mean.astype(np.float32), std.astype(np.float32), scale.astype(np.float32)


def prepare(args):
    episodes, meta = load_replayed_demos(args.h5)
    if args.max_episodes:
        episodes = episodes[:args.max_episodes]
    train_idx, val_idx = split_episodes(len(episodes), args.val_fraction, args.split_seed)
    out = Path(args.out_dir)
    source = dict(source="released PPO demos, replayed to state obs", source_h5=str(args.h5),
                  source_env_info=meta["env_info"], source_commit=meta.get("commit_info"),
                  split_seed=args.split_seed, val_fraction=args.val_fraction)
    for name, idx in (("demos_train", train_idx), ("demos_val", val_idx)):
        eps = [episodes[i] for i in idx]
        save_transitions(out / f"{name}.npz",
                         np.concatenate([e["obs"] for e in eps]),
                         np.concatenate([e["actions"] for e in eps]),
                         np.concatenate([np.full(len(e["actions"]), e["episode_id"]) for e in eps]),
                         dict(source, split=name, episode_ids=[e["episode_id"] for e in eps]))
    raw = np.concatenate([e["raw_actions"] for e in episodes])
    summary = dict(
        state_dim=int(episodes[0]["obs"].shape[1]), action_dim=int(raw.shape[1]),
        episodes_in_file=len(meta["episodes"]), episodes_replayed=len(episodes),
        train_episodes=len(train_idx), val_episodes=len(val_idx),
        raw_action_min=raw.min(0).round(4).tolist(), raw_action_max=raw.max(0).round(4).tolist(),
        fraction_raw_actions_outside_bounds=float((np.abs(raw) > 1).mean()),
        replayed_success_at_end=float(np.mean([e["success_at_end"] for e in episodes])),
    )
    (out / "demos_summary.json").write_text(json.dumps(summary, indent=2))
    print(json.dumps(summary, indent=2))


# --------------------------------------------------------------------------------------
# Models
# --------------------------------------------------------------------------------------

class BCPolicy(nn.Module):
    """MLP teacher: normalized state -> action in [-1, 1].

    Normalization statistics are registered buffers, so a checkpoint carries its own
    preprocessing and the policy accepts raw environment observations.
    """

    def __init__(self, state_dim, action_dim, state_mean, state_scale, hidden=256, depth=2, tanh_output=True):
        super().__init__()
        layers, width = [], state_dim
        for _ in range(depth):
            layers += [nn.Linear(width, hidden), nn.ReLU()]
            width = hidden
        layers.append(nn.Linear(width, action_dim))
        self.net = nn.Sequential(*layers)
        self.tanh_output = tanh_output
        self.register_buffer("state_mean", torch.as_tensor(state_mean, dtype=torch.float32))
        self.register_buffer("state_scale", torch.as_tensor(state_scale, dtype=torch.float32))
        self.config = dict(state_dim=state_dim, action_dim=action_dim, hidden=hidden, depth=depth, tanh_output=tanh_output)

    def forward(self, raw_state):
        out = self.net((raw_state - self.state_mean) / self.state_scale)
        return torch.tanh(out) if self.tanh_output else out

    @torch.no_grad()
    def act(self, raw_state):
        return self(raw_state).clamp(-1.0, 1.0)


def save_policy(policy: BCPolicy, path, extra=None):
    torch.save({"config": policy.config, "state_dict": policy.state_dict(), "extra": extra or {}}, path)


def load_policy(path, device="cpu") -> BCPolicy:
    ckpt = torch.load(path, map_location=device, weights_only=False)
    cfg = ckpt["config"]
    policy = BCPolicy(cfg["state_dim"], cfg["action_dim"], np.zeros(cfg["state_dim"]), np.ones(cfg["state_dim"]),
                      hidden=cfg["hidden"], depth=cfg["depth"], tanh_output=cfg["tanh_output"])
    policy.load_state_dict(ckpt["state_dict"])
    return policy.to(device).eval()


class ReleasedPPOActor(nn.Module):
    """Deterministic mean of ManiSkill's state PPO actor (examples/baselines/ppo/ppo.py)."""

    def __init__(self, state_dim, action_dim):
        super().__init__()
        self.actor_mean = nn.Sequential(
            nn.Linear(state_dim, 256), nn.Tanh(), nn.Linear(256, 256), nn.Tanh(),
            nn.Linear(256, 256), nn.Tanh(), nn.Linear(256, action_dim))

    @torch.no_grad()
    def act(self, raw_state):
        return self.actor_mean(raw_state).clamp(-1.0, 1.0)


def load_released_ppo(path, state_dim, action_dim, device="cpu"):
    weights = torch.load(path, map_location=device, weights_only=False)
    actor = ReleasedPPOActor(state_dim, action_dim)
    actor.load_state_dict({k: v for k, v in weights.items() if k.startswith("actor_mean.")})
    return actor.to(device).eval()


# --------------------------------------------------------------------------------------
# Rollouts (shared by evaluation and data collection)
# --------------------------------------------------------------------------------------

@torch.no_grad()
def rollout(env, max_steps, driver, seed, labeler=None, noise_std=0.0, generator=None, reset_options=None):
    """Run one batch of episodes, one per parallel env, for max_steps with no resets.

    driver picks the executed action; Gaussian noise (std on the [-1, 1] action scale) is
    added after labeling, so labels stay the labeler's clean actions. If labeler is given,
    returns obs [n, T, d_s] and labels [n, T, d_a] on the CPU, episode-major, and per-episode
    records (including first_success_step, -1 if never, and success_at_end).
    reset_options go to env.reset, e.g. dict(goal_xy=...) to fix PushTVarGoal-v1's goals.
    """
    obs, _ = env.reset(seed=seed) if reset_options is None else env.reset(seed=seed, options=reset_options)
    n = obs.shape[0]
    device = obs.device
    base = getattr(env, "base_env", None)
    if base is None:
        base = getattr(env, "unwrapped", None)   # raw ManiSkill env (no vector wrapper)
    # Per-episode start/goal records from the variable-goal envs, read right after reset.
    records = {name: getattr(base, name).clone() for name in EPISODE_RECORDS if hasattr(base, name)}
    if "goal_xy" not in records and hasattr(base, "goal_quadrant") and hasattr(base, "goal_tee"):
        records["goal_xy"] = base.goal_tee.pose.p[:, :2].clone()
    quadrant = records.get("goal_quadrant")
    success_once = torch.zeros(n, dtype=torch.bool, device=device)
    first_success = torch.full((n,), -1, dtype=torch.long, device=device)
    obs_buf, label_buf = [], []
    success = success_once
    for t in range(max_steps):
        o = obs.float()
        action = driver.act(o)
        if labeler is not None:
            label = action if labeler is driver else labeler.act(o)
            obs_buf.append(o.cpu())
            label_buf.append(label.cpu())
        if noise_std > 0:
            noise = torch.randn(action.shape, device=action.device, generator=generator)
            action = (action + noise_std * noise).clamp(-1.0, 1.0)
        obs, _, _, _, info = env.step(action)
        success = info["success"].bool()
        newly = (first_success < 0) & success
        first_success = torch.where(newly, torch.full_like(first_success, t + 1), first_success)
        success_once |= success
    steps = first_success[first_success > 0].float()
    result = {
        "episodes": int(n), "seed": int(seed), "noise_std": float(noise_std),
        "success_once": success_once.float().mean().item(),
        "success_at_end": success.float().mean().item(),
        "mean_steps_to_first_success": steps.mean().item() if len(steps) else None,
    }
    if quadrant is not None:
        result["by_goal_quadrant"] = {
            f"Q{q}": {"episodes": int(m.sum()),
                      "success_once": success_once[m].float().mean().item(),
                      "success_at_end": success[m].float().mean().item()}
            for q in range(1, 5) if (m := quadrant == q).any()}
    if "tip_ik_fallback" in records:
        result["tip_ik_fallback_rate"] = records["tip_ik_fallback"].float().mean().item()
    if labeler is not None:
        records["first_success_step"] = first_success
        records["success_at_end"] = success
        for name, value in records.items():
            result[f"{name}_per_episode"] = value.cpu().tolist()
    if labeler is None:
        return result
    return result, torch.stack(obs_buf, 1).numpy(), torch.stack(label_buf, 1).numpy()


# --------------------------------------------------------------------------------------
# Training
# --------------------------------------------------------------------------------------

@torch.no_grad()
def action_mse(policy, obs, actions, device, batch=65536):
    total, count = 0.0, 0
    for i in range(0, len(obs), batch):
        o = torch.as_tensor(obs[i:i + batch], device=device)
        a = torch.as_tensor(actions[i:i + batch], device=device)
        total += torch.nn.functional.mse_loss(policy(o), a, reduction="sum").item()
        count += a.numel()
    return total / max(count, 1)


def train(args):
    torch.manual_seed(args.seed)
    np.random.seed(args.seed)
    device = torch.device(args.device)
    out = Path(args.out)
    out.mkdir(parents=True, exist_ok=True)

    obs_np, act_np, ep_np = load_transitions(args.train_data)
    val_obs, val_act, _ = load_transitions([args.val_data])
    mean, std, scale = compute_state_stats(obs_np)
    data_info = dict(train_files=[str(p) for p in args.train_data], val_file=str(args.val_data),
                     train_transitions=int(len(obs_np)), train_episodes=int(len(np.unique(ep_np))),
                     val_transitions=int(len(val_obs)),
                     features_only_centered=np.flatnonzero(std < MIN_SCALED_STD).tolist())
    (out / "data_info.json").write_text(json.dumps(data_info, indent=2))
    print(f"Training on {data_info['train_transitions']:,} transitions from "
          f"{data_info['train_episodes']:,} episodes in {len(args.train_data)} file(s)", flush=True)

    train_obs = torch.as_tensor(obs_np, device=device)
    train_act = torch.as_tensor(act_np, device=device).clamp(-1.0, 1.0)
    if args.tanh_output:
        # tanh cannot reach exactly +-1; keep targets just inside the bound.
        train_act = train_act.clamp(-0.999, 0.999)
    state_dim, action_dim = train_obs.shape[1], train_act.shape[1]
    policy = BCPolicy(state_dim, action_dim, mean, scale,
                      hidden=args.hidden, depth=args.depth, tanh_output=args.tanh_output).to(device)
    opt = torch.optim.Adam(policy.parameters(), lr=args.lr)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=args.iters) if args.cosine else None
    gen = torch.Generator(device=device).manual_seed(args.seed)

    evaluator = Evaluator(num_envs=args.eval_episodes, device=device) if args.eval_every > 0 else None

    fields = ("iter", "train_mse", "val_mse", "lr", "success_once", "success_at_end", "seconds")
    log = (out / "train_log.csv").open("w")
    log.write(",".join(fields) + "\n")
    best = (-1.0, -1.0)
    start = time.time()
    running, samples = 0.0, 0
    for it in range(1, args.iters + 1):
        idx = torch.randint(0, len(train_obs), (args.batch_size,), device=device, generator=gen)
        loss = torch.nn.functional.mse_loss(policy(train_obs[idx]), train_act[idx])
        opt.zero_grad(set_to_none=True)
        loss.backward()
        opt.step()
        if sched:
            sched.step()
        if it % 100 == 0:
            running += loss.item()
            samples += 1

        last = it == args.iters
        if it % args.log_every == 0 or last:
            policy.eval()
            val_mse = action_mse(policy, val_obs, val_act.clip(-1, 1), device)
            row = dict(iter=it, train_mse=running / max(samples, 1), val_mse=val_mse,
                       lr=opt.param_groups[0]["lr"], success_once="", success_at_end="")
            running, samples = 0.0, 0
            if evaluator and (it % args.eval_every == 0 or last):
                res = evaluator.run(policy, seed=args.val_seed)
                row.update(success_once=res["success_once"], success_at_end=res["success_at_end"])
                save_policy(policy, out / f"ckpt_{it}.pt", {"iter": it, **res})
                key = (res["success_at_end"], res["success_once"])
                if key > best:
                    best = key
                    save_policy(policy, out / "best.pt", {"iter": it, **res})
                    (out / "best.json").write_text(json.dumps({"iter": it, **res}, indent=2))
            policy.train()
            row["seconds"] = round(time.time() - start, 1)
            log.write(",".join(str(row[k]) for k in fields) + "\n")
            log.flush()
            msg = f"iter {it:>7}  train_mse {row['train_mse']:.5f}  val_mse {val_mse:.5f}"
            if row["success_once"] != "":
                msg += f"  val success_once {row['success_once']:.0%}  success_at_end {row['success_at_end']:.0%}"
            print(msg, flush=True)
    log.close()
    save_policy(policy, out / "final.pt", {"iter": args.iters})
    if evaluator is None:
        save_policy(policy, out / "best.pt", {"iter": args.iters, "note": "no closed-loop eval; best = final"})
    else:
        evaluator.close()
    print(f"Done in {(time.time() - start) / 60:.1f} min. Best validation (end, once): {best}")


# --------------------------------------------------------------------------------------
# Simulator: audit, evaluation, collection
# --------------------------------------------------------------------------------------

def configure_env(env_id, goal_quadrants=None, start_quadrants=None):
    """Select the environment for this process (registration happens in make_env)."""
    global ENV_ID, ENV_KWARGS
    ENV_ID, ENV_KWARGS = env_id, {}
    if goal_quadrants:
        if env_id not in GOAL_ENV_IDS:
            raise ValueError("--goal-quadrants only applies to " + " and ".join(GOAL_ENV_IDS))
        ENV_KWARGS["goal_quadrants"] = tuple(goal_quadrants)
    if start_quadrants:
        if env_id != VARIABLE_GOAL_ENV_ID:
            raise ValueError("--start-quadrants only applies to " + VARIABLE_GOAL_ENV_ID)
        ENV_KWARGS["start_quadrants"] = tuple(start_quadrants)


def make_env(num_envs, obs_mode=OBS_MODE, **kwargs):
    import gymnasium as gym
    import mani_skill.envs  # noqa: F401  (registers environments)

    if ENV_ID == VARIABLE_GOAL_ENV_ID:
        import variable_goal_env  # registers PushT-VariableGoal-v1
        if variable_goal_env.PushTVariableGoalEnv is None:
            raise ImportError("variable_goal_env could not import ManiSkill; the environment is not registered")
    elif ENV_ID == SHARED_ENV_ID:
        import pusht_vargoal  # noqa: F401  (registers PushTVarGoal-v1; Jimmy's state_rl/pusht_vargoal.py)
    return gym.make(ENV_ID, num_envs=num_envs, obs_mode=obs_mode, control_mode=CONTROL_MODE,
                    sim_backend=SIM_BACKEND, render_mode="rgb_array", **ENV_KWARGS, **kwargs)


def run_ppo(argv):
    """Run ManiSkill's ppo.py in this process after registering the variable-goal environment.
    ppo.py writes runs/<exp-name>/ under the current directory."""
    import runpy
    import sys

    p = argparse.ArgumentParser(prog="pusht_state_bc.py ppo", allow_abbrev=False)
    p.add_argument("--script", required=True, help="path to ManiSkill examples/baselines/ppo/ppo.py")
    known, rest = p.parse_known_args(argv)
    import variable_goal_env  # noqa: F401  (registers PushT-VariableGoal-v1 for ppo.py's gym.make)
    sys.argv = [known.script] + rest
    runpy.run_path(known.script, run_name="__main__")


def episode_length(env):
    """Max episode steps. Vectorized GPU envs leave env.spec.max_episode_steps as None,
    so use ManiSkill's own lookup (as its PPO baseline does), with PushT-v1's registered 100 as fallback."""
    from mani_skill.utils import gym_utils

    steps = gym_utils.find_max_episode_steps_value(env)
    return int(steps) if steps is not None else 100


def make_vector_env(num_envs, video_dir=None, **kwargs):
    """Fixed-length episodes: no auto-reset (it would replace the final step's info) and
    terminations ignored, matching the evaluation convention of ManiSkill's PPO baseline."""
    from mani_skill.vector.wrappers.gymnasium import ManiSkillVectorEnv

    env = make_env(num_envs, **kwargs)
    max_steps = episode_length(env)
    if video_dir is not None:
        from mani_skill.utils.wrappers.record import RecordEpisode
        env = RecordEpisode(env, output_dir=str(video_dir), save_trajectory=False,
                            max_steps_per_video=max_steps, video_fps=30)
    return ManiSkillVectorEnv(env, num_envs, auto_reset=False, ignore_terminations=True, record_metrics=True), max_steps


def flatten_with_names(tree, prefix=""):
    """Flatten a nested state dict in insertion order, as ManiSkill does; return names and widths."""
    fields = []
    for key, value in tree.items():
        name = f"{prefix}{key}"
        if isinstance(value, dict):
            fields += flatten_with_names(value, name + "/")
        else:
            width = 1 if value.ndim == 1 else int(value.shape[-1])
            fields.append((name, width))
    return fields


def audit(args):
    import mani_skill

    out = Path(args.out)
    out.mkdir(parents=True, exist_ok=True)
    env = make_env(1)
    try:
        obs, info = env.reset(seed=0)
        tree = env.unwrapped.get_obs(info, unflattened=True)
        fields = flatten_with_names(tree)
        width = sum(w for _, w in fields)
        assert obs.shape == (1, width), f"Flat obs {tuple(obs.shape)} does not match fields totalling {width}"
        offsets, start = {}, 0
        for name, w in fields:
            offsets[name] = [start, start + w]
            start += w
        if not any("obj_pose" in n for n, _ in fields):
            raise AssertionError("State observation lacks obj_pose; this is not the full-state setting")
        space = env.unwrapped.single_action_space
        contract = {
            "env_id": ENV_ID, "env_kwargs": {k: list(v) for k, v in ENV_KWARGS.items()}, "obs_mode": OBS_MODE, "control_mode": CONTROL_MODE, "sim_backend": SIM_BACKEND,
            "mani_skill_version": getattr(mani_skill, "__version__", "unknown"),
            "state_dim": width, "state_fields_in_order": offsets,
            "action_dim": int(np.prod(space.shape)),
            "action_low": space.low.tolist(), "action_high": space.high.tolist(),
            "control_freq": getattr(env.unwrapped, "control_freq", None),
            "sim_freq": getattr(env.unwrapped, "sim_freq", None),
            "max_episode_steps": episode_length(env),
            "robot_uids": str(getattr(env.unwrapped, "robot_uids", None)),
            "success_rule": "evaluate(): T overlap with goal >= intersection_thresh",
            "intersection_thresh": getattr(env.unwrapped, "intersection_thresh", None),
        }
        if not (np.allclose(space.low, -1) and np.allclose(space.high, 1)):
            print("WARNING: action space is not [-1, 1]; the BC tanh head assumes normalized actions.")
        (out / "observation_contract.json").write_text(json.dumps(contract, indent=2))
        print(json.dumps(contract, indent=2))
    finally:
        env.close()


def check_env(args):
    """Check the success test against each episode's own goal, without stepping physics.

    Puts a stand-in T exactly on every episode's goal pose and runs the environment's overlap
    computation: all must count as success. Then shifts it 5 cm: none may. With PushT-v1's single
    fixed goal transform, varied goals would fail the first check. Also reports goal ranges and quadrants.
    """
    import types

    env = make_env(args.num_envs)
    try:
        env.reset(seed=args.seed)
        base = env.unwrapped
        goal_p, goal_q = base.goal_tee.pose.p.clone(), base.goal_tee.pose.q.clone()
        real_tee = base.tee

        def overlap(p):
            base.tee = types.SimpleNamespace(pose=types.SimpleNamespace(p=p, q=goal_q))
            try:
                return base.pseudo_render_intersection()
            finally:
                base.tee = real_tee

        at_goal = overlap(goal_p)
        shifted = goal_p.clone()
        shifted[:, 0] += 0.05
        off_goal = overlap(shifted)
        thresh = base.intersection_thresh
        quadrant = getattr(base, "goal_quadrant", None)
        report = {
            "env_id": ENV_ID, "num_envs": args.num_envs,
            "goal_x_range": [goal_p[:, 0].min().item(), goal_p[:, 0].max().item()],
            "goal_y_range": [goal_p[:, 1].min().item(), goal_p[:, 1].max().item()],
            "goal_quadrant_counts": ({f"Q{q}": int((quadrant == q).sum()) for q in range(1, 5)}
                                     if quadrant is not None else None),
            "overlap_with_T_on_goal_min": at_goal.min().item(),
            "overlap_with_T_shifted_5cm_max": off_goal.max().item(),
            "intersection_thresh": thresh,
        }
        passed = bool((at_goal >= thresh).all() and (off_goal < thresh).all())
        if hasattr(base, "tip_start_xy") and getattr(base, "random_tip_start", False):
            # Where the stick tip actually is after reset, versus where IK placed it.
            tcp_xy = base.agent.tcp.pose.p[:, :2]
            placed = ~base.tip_ik_fallback
            gap = torch.linalg.norm(tcp_xy - base.tip_start_xy, dim=1)
            tee_dist = torch.linalg.norm(tcp_xy - base.tee.pose.p[:, :2], dim=1)
            tip_q = base.tip_start_quadrant
            report.update({
                "tip_x_range": [tcp_xy[:, 0].min().item(), tcp_xy[:, 0].max().item()],
                "tip_y_range": [tcp_xy[:, 1].min().item(), tcp_xy[:, 1].max().item()],
                "tip_start_quadrant_counts": {f"Q{q}": int((tip_q == q).sum()) for q in range(1, 5)},
                "tee_start_quadrant_counts": {f"Q{q}": int((base.tee_start_quadrant == q).sum()) for q in range(1, 5)},
                "tip_ik_fallback_rate": base.tip_ik_fallback.float().mean().item(),
                "tip_grid_reachable_fraction": base._ik["valid_fraction"] if getattr(base, "_ik", None) else None,
                "tip_ik_error_max_m": base.tip_ik_error.max().item(),
                "tip_vs_ik_gap_max_m": gap[placed].max().item() if placed.any() else None,
                "tip_height_range_m": [base.agent.tcp.pose.p[:, 2].min().item(), base.agent.tcp.pose.p[:, 2].max().item()],
                "tip_to_tee_min_m": tee_dist.min().item(),
                "goal_vs_tip_quadrant_agreement": (tip_q == quadrant).float().mean().item() if quadrant is not None else None,
            })
            # Placement must match the simulator's own kinematics (catches a wrong base offset or chain)
            # and fall back rarely.
            passed = passed and report["tip_ik_fallback_rate"] <= 0.05 and (
                report["tip_vs_ik_gap_max_m"] is None or report["tip_vs_ik_gap_max_m"] < 0.02)
        report["passed"] = passed
    finally:
        env.close()
    print(json.dumps(report, indent=2))
    if args.out:
        Path(args.out).write_text(json.dumps(report, indent=2))
    if not report["passed"]:
        raise SystemExit("Environment check FAILED: success is not measured against each episode's goal.")


class Evaluator:
    """Closed-loop evaluation on a fixed batch of seeded initial states."""

    def __init__(self, num_envs, device, video_dir=None):
        self.env, self.max_steps = make_vector_env(num_envs, video_dir=video_dir, reconfiguration_freq=1)
        self.num_envs = num_envs
        self.device = device

    def run(self, policy, seed):
        return rollout(self.env, self.max_steps, policy, seed)

    def close(self):
        self.env.close()


def load_expert(path, env, device):
    spaces = env if hasattr(env, "single_observation_space") else env.unwrapped
    state_dim = int(np.prod(spaces.single_observation_space.shape))
    action_dim = int(np.prod(spaces.single_action_space.shape))
    return load_released_ppo(path, state_dim, action_dim, device)


def eval_per_quadrant(env, max_steps, policy, base_seed, quadrants=(1, 2, 3, 4), bar=None):
    """Shared per-quadrant protocol (state_rl/pusht_state_ppo.py): for each quadrant q, restrict goals
    to q and run one batch of episodes with seed base_seed + 1000 * q."""
    base = env.unwrapped
    original = tuple(base.goal_quadrants)
    out = {}
    try:
        for q in quadrants:
            base.goal_quadrants = (q,)
            r = rollout(env, max_steps, policy, base_seed + 1000 * q)
            if not bool((base.goal_quadrant == q).all()):
                raise AssertionError(f"goals left the requested quadrant Q{q}")
            r.pop("by_goal_quadrant", None)
            out[f"Q{q}"] = r
    finally:
        base.goal_quadrants = original
    once = [out[f"Q{q}"]["success_once"] for q in quadrants]
    end = [out[f"Q{q}"]["success_at_end"] for q in quadrants]
    out.update(worst_success_once=min(once), mean_success_once=float(np.mean(once)),
               mean_success_at_end=float(np.mean(end)))
    if bar is not None:
        out.update(freeze_bar=bar, passes_bar=all(v >= bar for v in once))
    return out


def eval_quadrants(args):
    """Each checkpoint, quadrant by quadrant, on a raw (unwrapped, not reconfigured) env, as in
    state_rl/pusht_state_ppo.py, so BC and PPO teachers are scored on the same episodes."""
    device = torch.device(args.device)
    env = make_env(args.episodes)
    if not hasattr(env.unwrapped, "goal_quadrants"):
        env.close()
        raise SystemExit(f"{ENV_ID} has no goal_quadrants; eval-quadrants needs a variable-goal env")
    rows = []
    try:
        max_steps = episode_length(env)
        for ckpt in args.checkpoint:
            policy = load_expert(ckpt, env, device) if args.ppo else load_policy(ckpt, device)
            r = eval_per_quadrant(env, max_steps, policy, args.seed, bar=args.bar)
            rows.append({"checkpoint": str(ckpt), "env_id": ENV_ID, "episodes_per_quadrant": args.episodes,
                         "base_seed": args.seed, **r})
            print(f"{Path(ckpt).name:>24}  worst {r['worst_success_once']:.0%}  mean {r['mean_success_once']:.0%}  "
                  + "  ".join(f"Q{q} {r[f'Q{q}']['success_once']:.0%}/{r[f'Q{q}']['success_at_end']:.0%}"
                              for q in (1, 2, 3, 4)), flush=True)
    finally:
        env.close()
    if args.out:
        Path(args.out).write_text(json.dumps(rows if len(rows) > 1 else rows[0], indent=2))
    return rows


def evaluate(args):
    device = torch.device(args.device)
    video_dir = Path(args.video_dir) if args.video_dir else None
    evaluator = Evaluator(args.episodes, device, video_dir=video_dir)
    try:
        if args.ppo:
            policy = load_expert(args.checkpoint, evaluator.env, device)
        else:
            policy = load_policy(args.checkpoint, device)
        res = evaluator.run(policy, seed=args.seed)
    finally:
        evaluator.close()
    res["checkpoint"] = str(args.checkpoint)
    print(json.dumps(res, indent=2))
    if args.out:
        Path(args.out).write_text(json.dumps(res, indent=2))
    return res


def collect(args):
    """Roll out a driver and label every visited state with the expert.

    driver=expert (optionally with action noise) gives more, and slightly off-course,
    expert demonstrations. driver=bc gives one round of DAgger data: states the BC policy
    actually reaches, labeled with what the expert would do there.
    """
    device = torch.device(args.device)
    env, max_steps = make_vector_env(args.num_envs)
    try:
        expert = load_expert(args.expert, env, device)
        driver = expert if args.driver == "expert" else load_policy(args.driver_checkpoint, device)
        noise_levels = args.noise_std or [0.0]
        gen = torch.Generator(device=device).manual_seed(args.seed_base)
        batches = math.ceil(args.episodes / args.num_envs)
        all_obs, all_labels, per_batch, all_noise = [], [], [], []
        per_episode = {}
        for b in range(batches):
            sigma = noise_levels[b % len(noise_levels)]
            res, obs, labels = rollout(env, max_steps, driver, seed=args.seed_base + b,
                                       labeler=expert, noise_std=sigma, generator=gen)
            all_obs.append(obs)
            all_labels.append(labels)
            for key in [k for k in res if k.endswith("_per_episode")]:
                per_episode.setdefault(key, []).extend(res.pop(key))
            all_noise += [sigma] * obs.shape[0]
            per_batch.append(res)
            print(f"batch {b + 1}/{batches}  noise {sigma:.2f}  driver success_once {res['success_once']:.0%}  "
                  f"success_at_end {res['success_at_end']:.0%}", flush=True)
    finally:
        env.close()
    obs = np.concatenate(all_obs)[:args.episodes]       # [episodes, T, d_s]
    labels = np.concatenate(all_labels)[:args.episodes]
    n, T = obs.shape[:2]
    meta = dict(
        driver=args.driver, driver_checkpoint=args.driver_checkpoint or None, expert=str(args.expert),
        labels="expert deterministic action, clipped to [-1, 1]", noise_std_cycle=noise_levels,
        env_id=ENV_ID, env_kwargs={k: list(v) for k, v in ENV_KWARGS.items()},
        seed_base=args.seed_base, num_envs=args.num_envs, steps_per_episode=int(T), batches=per_batch,
        driver_success_once=float(np.mean([r["success_once"] for r in per_batch])),
        driver_success_at_end=float(np.mean([r["success_at_end"] for r in per_batch])),
        noise_std_per_episode=all_noise[:n],
    )
    for key, values in per_episode.items():
        meta[key] = values[:n]
    meta = save_transitions(args.out, obs.reshape(n * T, -1), labels.reshape(n * T, -1),
                            np.repeat(np.arange(n), T), meta)
    print(json.dumps({k: v for k, v in meta.items() if not k.endswith("per_episode") and k != "batches"}, indent=2))


# --------------------------------------------------------------------------------------
# CLI
# --------------------------------------------------------------------------------------

def main(argv=None):
    import sys

    argv = sys.argv[1:] if argv is None else list(argv)
    if argv and argv[0] == "ppo":
        return run_ppo(argv[1:])

    p = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    sub = p.add_subparsers(dest="cmd", required=True)
    sub.add_parser("ppo", help="run ManiSkill ppo.py on the variable-goal env (see run_ppo)")
    default_device = "cuda" if torch.cuda.is_available() else "cpu"
    env_opts = argparse.ArgumentParser(add_help=False)
    env_opts.add_argument("--env-id", default=VARIABLE_GOAL_ENV_ID)
    env_opts.add_argument("--goal-quadrants", type=int, nargs="+", default=None,
                          help="goal quadrants to sample (variable-goal env only); default all four")
    env_opts.add_argument("--start-quadrants", type=int, nargs="+", default=None,
                          help="stick-tip start quadrants to sample (variable-goal env only); default all four")

    a = sub.add_parser("audit", parents=[env_opts])
    a.add_argument("--out", required=True)

    ce = sub.add_parser("check-env", parents=[env_opts])
    ce.add_argument("--num-envs", type=int, default=256)
    ce.add_argument("--seed", type=int, default=0)
    ce.add_argument("--out", default="")

    pr = sub.add_parser("prepare")
    pr.add_argument("--h5", required=True)
    pr.add_argument("--out-dir", required=True)
    pr.add_argument("--val-fraction", type=float, default=0.1)
    pr.add_argument("--split-seed", type=int, default=0)
    pr.add_argument("--max-episodes", type=int, default=0, help="0 = use all episodes")

    c = sub.add_parser("collect", parents=[env_opts])
    c.add_argument("--expert", required=True, help="PPO checkpoint (ManiSkill ppo.py format) used for labels")
    c.add_argument("--driver", choices=("expert", "bc"), default="expert")
    c.add_argument("--driver-checkpoint", default="", help="BC checkpoint when --driver bc")
    c.add_argument("--episodes", type=int, required=True)
    c.add_argument("--num-envs", type=int, default=1024)
    c.add_argument("--noise-std", type=float, nargs="*", default=[0.0],
                   help="action noise std per batch, cycled; labels stay noise-free")
    c.add_argument("--seed-base", type=int, required=True)
    c.add_argument("--out", required=True)
    c.add_argument("--device", default=default_device)

    t = sub.add_parser("train", parents=[env_opts])
    t.add_argument("--train-data", nargs="+", required=True)
    t.add_argument("--val-data", required=True)
    t.add_argument("--out", required=True)
    t.add_argument("--seed", type=int, default=0)
    t.add_argument("--iters", type=int, default=100_000)
    t.add_argument("--batch-size", type=int, default=1024)
    t.add_argument("--lr", type=float, default=3e-4)
    t.add_argument("--cosine", action=argparse.BooleanOptionalAction, default=True)
    t.add_argument("--hidden", type=int, default=256)
    t.add_argument("--depth", type=int, default=2)
    t.add_argument("--tanh-output", action=argparse.BooleanOptionalAction, default=True)
    t.add_argument("--log-every", type=int, default=1000)
    t.add_argument("--eval-every", type=int, default=10_000, help="0 disables closed-loop eval")
    t.add_argument("--eval-episodes", type=int, default=100)
    t.add_argument("--val-seed", type=int, default=1001)
    t.add_argument("--device", default=default_device)

    eq = sub.add_parser("eval-quadrants", parents=[env_opts])
    eq.add_argument("--checkpoint", nargs="+", required=True)
    eq.add_argument("--ppo", action="store_true", help="checkpoints are ManiSkill ppo.py checkpoints")
    eq.add_argument("--episodes", type=int, default=100, help="per quadrant")
    eq.add_argument("--seed", type=int, default=1001, help="quadrant q uses seed + 1000 * q")
    eq.add_argument("--bar", type=float, default=0.80, help="freeze bar: success-once in every quadrant")
    eq.add_argument("--out", default="")
    eq.add_argument("--device", default=default_device)

    e = sub.add_parser("eval", parents=[env_opts])
    e.add_argument("--checkpoint", required=True)
    e.add_argument("--ppo", "--released-ppo", dest="ppo", action="store_true",
                   help="checkpoint is a ManiSkill ppo.py checkpoint (released or our own)")
    e.add_argument("--episodes", type=int, default=100)
    e.add_argument("--seed", type=int, default=2001)
    e.add_argument("--out", default="")
    e.add_argument("--video-dir", default="")
    e.add_argument("--device", default=default_device)

    args = p.parse_args(argv)
    if args.cmd == "collect" and args.driver == "bc" and not args.driver_checkpoint:
        p.error("--driver bc needs --driver-checkpoint")
    if hasattr(args, "env_id"):
        configure_env(args.env_id, args.goal_quadrants, args.start_quadrants)
    {"audit": audit, "check-env": check_env, "prepare": prepare, "collect": collect, "train": train,
     "eval": evaluate, "eval-quadrants": eval_quadrants}[args.cmd](args)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /content/pusht_state_bc/pusht_vargoal.py
# Copied unchanged from jimmy-state-rl:state_rl/pusht_vargoal.py at c87d8ae
"""Variable-goal Push-T: ManiSkill PushT-v1 with a different target position per episode.

Registers ``PushTVarGoal-v1``. Import this module before ``gym.make``.

What changes relative to PushT-v1 (ManiSkill 3.0.1, commit a4a4f92):
  * Goal T position is sampled per episode from four quadrants around ``QUAD_ORIGIN``.
    Goal orientation stays fixed at PushT-v1's 5*pi/3.
  * The success check (``pseudo_render_intersection``) uses a goal transform per
    environment, recomputed at every reset. In PushT-v1 it is one fixed matrix, so a
    moved goal would silently be scored against the old target.
  * The T block starts anywhere in ``TEE_BOX`` (independent of the goal quadrant), with
    its distance to the goal limited to ``START_GOAL_DIST``.
  * The stick tip starts at a random point in ``TIP_BOX``, drawn from a precomputed
    library of arm poses (``assets/tip_start_qpos.npz``, built by make_tip_starts.py),
    at least ``TIP_TEE_MIN`` from the block.

What does NOT change: observation (31 numbers, same order; ``extra/goal_pos`` now
varies), action space and controller, 100-step episodes, the dense reward (it already
reads the goal T's live pose), and the success rule (overlap >= 0.90).

Quadrants, with dx = x - QUAD_ORIGIN[0] (positive = away from the robot base) and y:
  Q1: dx > 0, y > 0    Q2: dx < 0, y > 0    Q3: dx < 0, y < 0    Q4: dx > 0, y < 0
A band of ``AXIS_GAP`` around each axis is excluded so neighbouring quadrants do not
share near-identical goals.

Usage:
  env = gym.make("PushTVarGoal-v1", obs_mode="state", control_mode="pd_joint_delta_pos",
                 goal_quadrants=(1, 2, 3, 4))
  env.reset(seed=0)                                    # random goals in those quadrants
  env.reset(options=dict(goal_xy=torch.tensor([[-0.2, 0.1]])))  # fixed goal(s), [b, 2]
  env.unwrapped.goal_quadrant                          # [num_envs] quadrant of current goal
"""
from pathlib import Path

import numpy as np
import torch

from mani_skill.envs.tasks.tabletop.push_t import PushTEnv
from mani_skill.utils.registration import register_env
from mani_skill.utils.structs.pose import Pose

# ---- Workspace (world frame, metres). Robot base is at x = -0.615. ----------------
# Measured stick-tip reach at table height: x in about [-0.40, 0.05], y in about
# [-0.45, 0.45] (see README). Regions below keep >= ~0.1 m margin inside that.
QUAD_ORIGIN = (-0.156, 0.0)          # PushT-v1's goal x; centred in y
GOAL_HALF = (0.10, 0.20)             # goals in origin +- these
AXIS_GAP = (0.02, 0.03)              # excluded band around each quadrant axis
TEE_BOX = ((-0.30, -0.02), (-0.28, 0.28))   # T centre-of-mass start region
TIP_BOX = ((-0.36, 0.00), (-0.30, 0.30))    # stick-tip start region
START_GOAL_DIST = (0.08, 0.30)       # allowed T-start-to-goal distance
TIP_TEE_MIN = 0.15                   # stick tip starts at least this far from the T
N_CANDIDATES = 64                    # rejection-sampling candidates per reset

_QUAD_SIGNS = {1: (1.0, 1.0), 2: (-1.0, 1.0), 3: (-1.0, -1.0), 4: (1.0, -1.0)}
_DEFAULT_TIP_FILE = Path(__file__).parent / "assets" / "tip_start_qpos.npz"


def quadrant_of(goal_xy: torch.Tensor) -> torch.Tensor:
    """Quadrant (1-4) of goal positions [b, 2]."""
    dx = goal_xy[:, 0] - QUAD_ORIGIN[0]
    dy = goal_xy[:, 1] - QUAD_ORIGIN[1]
    q = torch.full_like(dx, 0, dtype=torch.long)
    q[(dx > 0) & (dy > 0)] = 1
    q[(dx < 0) & (dy > 0)] = 2
    q[(dx < 0) & (dy < 0)] = 3
    q[(dx > 0) & (dy < 0)] = 4
    return q


def sample_goals(b: int, quadrants, device=None) -> torch.Tensor:
    """Uniform goal positions [b, 2] in the given quadrants (each quadrant equally likely)."""
    quads = torch.tensor(list(quadrants), device=device)
    pick = quads[torch.randint(len(quads), (b,), device=device)]
    signs = torch.tensor([_QUAD_SIGNS[int(k)] for k in range(1, 5)], device=device)[pick - 1]
    lo = torch.tensor(AXIS_GAP, device=device)
    hi = torch.tensor(GOAL_HALF, device=device)
    mag = lo + torch.rand(b, 2, device=device) * (hi - lo)
    return torch.tensor(QUAD_ORIGIN, device=device) + signs * mag


def _uniform_box(shape, box, device):
    lo = torch.tensor([box[0][0], box[1][0]], device=device)
    hi = torch.tensor([box[0][1], box[1][1]], device=device)
    return lo + torch.rand(*shape, 2, device=device) * (hi - lo)


def _first_valid(valid: torch.Tensor) -> torch.Tensor:
    """Index of the first True per row of [b, K]; 0 if none (vanishingly rare with K=64)."""
    return valid.float().argmax(dim=1)


@register_env("PushTVarGoal-v1", max_episode_steps=100)
class PushTVarGoalEnv(PushTEnv):
    def __init__(self, *args, goal_quadrants=(1, 2, 3, 4), tip_start_file=None, **kwargs):
        assert set(goal_quadrants) <= {1, 2, 3, 4} and len(goal_quadrants) > 0
        self.goal_quadrants = tuple(int(q) for q in goal_quadrants)
        lib = np.load(tip_start_file or _DEFAULT_TIP_FILE)
        self._tip_qpos_np = lib["qpos"]
        self._tip_xy_np = lib["tip_xy"]
        super().__init__(*args, **kwargs)

    def _load_scene(self, options: dict):
        super()._load_scene(options)
        # One goal transform per environment instead of PushT-v1's single matrix.
        self.world_to_goal_trans = self.world_to_goal_trans.unsqueeze(0).repeat(self.num_envs, 1, 1)
        self.goal_quadrant = torch.zeros(self.num_envs, dtype=torch.long, device=self.device)
        self.tip_qpos_lib = torch.as_tensor(self._tip_qpos_np, device=self.device)
        self.tip_xy_lib = torch.as_tensor(self._tip_xy_np, device=self.device)

    def _goal_world_to_goal(self, goal_xy: torch.Tensor) -> torch.Tensor:
        b = goal_xy.shape[0]
        c, s = np.cos(self.goal_z_rot), np.sin(self.goal_z_rot)
        g = torch.zeros(b, 3, 3, device=goal_xy.device)
        g[:, 0, 0], g[:, 0, 1], g[:, 1, 0], g[:, 1, 1] = c, -s, s, c
        g[:, 2, 2] = 1.0
        g[:, 0:2, 2] = goal_xy
        return torch.linalg.inv(g)

    def _initialize_episode(self, env_idx: torch.Tensor, options: dict):
        # Parent sets up the table, default goal/T and robot rest pose; we override all three.
        super()._initialize_episode(env_idx, options)
        with torch.device(self.device):
            b = len(env_idx)

            # ---- goal ----
            goal_xy = (options or {}).get("goal_xy")
            if goal_xy is None:
                goal_xy = sample_goals(b, self.goal_quadrants, self.device)
            else:
                goal_xy = torch.as_tensor(goal_xy, dtype=torch.float32, device=self.device).reshape(-1, 2)
                if goal_xy.shape[0] == 1 and b > 1:
                    goal_xy = goal_xy.expand(b, 2)
                elif goal_xy.shape[0] == self.num_envs and b < self.num_envs:
                    goal_xy = goal_xy[env_idx]
                assert goal_xy.shape[0] == b, "goal_xy must be [1, 2], [b, 2] or [num_envs, 2]"
            goal_p = torch.zeros(b, 3)
            goal_p[:, :2] = goal_xy
            goal_p[:, 2] = 1e-3
            goal_q = torch.tensor([np.cos(self.goal_z_rot / 2), 0.0, 0.0, np.sin(self.goal_z_rot / 2)],
                                  dtype=torch.float32).expand(b, 4)
            self.goal_tee.set_pose(Pose.create_from_pq(p=goal_p, q=goal_q))
            self.world_to_goal_trans[env_idx] = self._goal_world_to_goal(goal_xy)
            self.goal_quadrant[env_idx] = quadrant_of(goal_xy)

            # ---- T block start: independent of goal quadrant, distance-limited ----
            cand = _uniform_box((b, N_CANDIDATES), TEE_BOX, self.device)          # [b, K, 2]
            d = (cand - goal_xy[:, None, :]).norm(dim=-1)
            pick = _first_valid((d >= START_GOAL_DIST[0]) & (d <= START_GOAL_DIST[1]))
            tee_xy = cand[torch.arange(b), pick]
            tee_p = torch.zeros(b, 3)
            tee_p[:, :2] = tee_xy
            tee_p[:, 2] = 0.04 / 2 + 1e-3
            ang = torch.rand(b) * (2 * torch.pi)
            tee_q = torch.zeros(b, 4)
            tee_q[:, 0], tee_q[:, 3] = (ang / 2).cos(), (ang / 2).sin()
            self.tee.set_pose(Pose.create_from_pq(p=tee_p, q=tee_q))

            # ---- stick tip start: library pose away from the T ----
            n_lib = self.tip_xy_lib.shape[0]
            idx = torch.randint(n_lib, (b, N_CANDIDATES))
            dt = (self.tip_xy_lib[idx] - tee_xy[:, None, :]).norm(dim=-1)
            pick = _first_valid(dt >= TIP_TEE_MIN)
            qpos = self.tip_qpos_lib[idx[torch.arange(b), pick]]
            qpos = qpos + torch.randn_like(qpos) * self.robot_init_qpos_noise
            self.agent.reset(qpos)


In [ ]:
# Jimmy's arm-pose library (base64), from jimmy-state-rl at c87d8ae
TIP_LIBRARY = ("...
""")
data = base64.b64decode("".join(TIP_LIBRARY[1].split()))
assert hashlib.sha256(data).hexdigest() == TIP_LIBRARY[0], "tip_start_qpos.npz: SHA-256 mismatch"
(WORK_ROOT / "assets").mkdir(exist_ok=True)
(WORK_ROOT / "assets" / "tip_start_qpos.npz").write_bytes(data)
ICL = WORK_ROOT / "pusht_icl.py"
for name in ("pusht_icl.py", "pusht_state_bc.py", "pusht_vargoal.py"):
    shutil.copy2(WORK_ROOT / name, OUTPUT_ROOT / name)
TEACHERS = ["--ppo-teacher", PPO_TEACHER, "--bc-teacher", BC_TEACHER]
run([TRAIN_PY, ICL, "--help"], show=r"^\s+(collect|train|offline|eval) ")

## 4. Demonstrations

Noise-free rollouts of both teachers: for every goal of every split, 10 episodes per teacher from different starts, with the same
goals for both teachers. Saved as `{split}_{teacher}.npz` (states with the goal, so the data stays useful for other experiments;
the goal is removed when prompts are built). Cached on Drive under `icl_data_v1/`; a later session copies them instead of re-collecting.

In [ ]:
LOCAL_DATA = Path("/content") / DATA_TAG
DRIVE_DATA = DRIVE_BASE / DATA_TAG
expected = [f"{s}_{t}.npz" for s in SPLITS for t in ("ppo", "bc")]
if all((DRIVE_DATA / f).exists() for f in expected):
    print("Copying cached demonstrations from", DRIVE_DATA)
    shutil.copytree(DRIVE_DATA, LOCAL_DATA, dirs_exist_ok=True)
else:
    run([TRAIN_PY, ICL, "collect", *TEACHERS, "--splits", *SPLITS, "--goals", *GOALS,
         "--episodes", EPISODES_PER_GOAL, "--out", LOCAL_DATA], show=r"batch \d+/\d+|\"success_once\"|exists")
    shutil.copytree(LOCAL_DATA, DRIVE_DATA, dirs_exist_ok=True)
for f in expected:
    meta = json.loads((LOCAL_DATA / f).with_suffix(".json").read_text())
    print(f"{f:<18} {meta['goals']:>5} goals x {meta['episodes_per_goal']} episodes  "
          f"success once {meta['success_once']:.0%}  at end {meta['success_at_end']:.0%}")

## 5. Pilot: 3,000 steps

Measures training speed (it/s and the ETA for the full run are printed) and checks that the model starts to use its context:
on validation goals, action error with **correct-goal** demonstrations should fall below **wrong-goal** and **no** demonstrations.
Then a small closed-loop check (64 validation goals, 2 starts each, PPO demonstrations, k = 0 and 4). Expect weak control this early;
the action-error gap is the signal. Stop here if correct and wrong are indistinguishable, and report back.

In [ ]:
PILOT_DIR = OUTPUT_ROOT / "pilot"
common = ["--data-dir", LOCAL_DATA, "--ks", *KS, "--stride", STRIDE, "--batch-size", BATCH_SIZE, "--lr", LR, "--seed", SEED]
run([TRAIN_PY, ICL, "train", *common, "--out", PILOT_DIR, "--iters", PILOT_ITERS, "--eval-every", 1000, "--log-every", 250],
    show=r"^(train|parameters|iter|Done)")
run([TRAIN_PY, ICL, "eval", *TEACHERS, "--checkpoint", PILOT_DIR / "final.pt", "--data-dir", LOCAL_DATA, "--splits", "val",
     "--context-teachers", "ppo", "--ks", 0, 4, "--stride", STRIDE, "--goals", 64, "--starts", 2,
     "--out", PILOT_DIR / "closed_loop_val.json"], show=r"^val ")

## 6. Full training

100,000 steps (about 1–3 hours; the pilot's it/s gives the real figure). The run lives in a fixed Drive folder
(`icl_runs/transformer_v1_seed0/`) with `last.pt` saved every 2,500 steps: **if Colab disconnects, rerun the whole notebook** and it continues
where it stopped (demonstrations come from the Drive cache). `best.pt` has the lowest validation action error with correct-goal demonstrations.

In [ ]:
run([TRAIN_PY, ICL, "train", *common, "--out", TRAIN_DIR, "--iters", ITERS, "--eval-every", EVAL_EVERY, "--log-every", 500],
    show=r"^(train|parameters|resumed|iter \d*(000|500) |Done)")

## 7. Training curves

Validation action error per context size: correct-goal (solid) vs wrong-goal (dashed) demonstrations, and no context.

In [ ]:
import matplotlib.pyplot as plt

rows = [json.loads(l) for l in (TRAIN_DIR / "train_log.jsonl").read_text().splitlines()]
it = [r["iter"] for r in rows]
fig, ax = plt.subplots(figsize=(8, 4))
for i, k in enumerate(map(str, KS)):
    colour = f"C{i}"
    if k == "0":
        ax.plot(it, [r["val"][k]["none"] for r in rows], color=colour, label="k=0 (no context)")
    else:
        ax.plot(it, [r["val"][k]["correct"] for r in rows], color=colour, label=f"k={k} correct")
        ax.plot(it, [r["val"][k]["wrong"] for r in rows], color=colour, ls="--", alpha=0.6)
ax.set(xlabel="iteration", ylabel="validation action MSE", yscale="log", title="Dashed: wrong-goal demonstrations")
ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(OUTPUT_ROOT / "training_curves.png", dpi=120)
plt.show()

## 8. Final evaluation (frozen weights)

`best.pt` on goals it never trained on: `test_in` (Q1–Q3) and `test_q4` (held-out quadrant).
- **Action error** (teacher-forced) per context size, correct-goal vs wrong-goal demonstrations.
- **Closed loop:** 128 goals × 4 new starts per split, for PPO and BC demonstrations, k ∈ {0, 1, 2, 4, 8}, correct-goal vs wrong-goal
  (k = 0 has no context). The teachers themselves (which see the goal) run on the same starts as the reference.

In [ ]:
EVAL_DIR = OUTPUT_ROOT / "final_eval"
EVAL_DIR.mkdir(exist_ok=True)
BEST = TRAIN_DIR / "best.pt"
run([TRAIN_PY, ICL, "offline", "--checkpoint", BEST, "--data-dir", LOCAL_DATA, "--splits", "val", "test_in", "test_q4",
     "--ks", *KS, "--stride", STRIDE, "--out", EVAL_DIR / "action_error.json"])
run([TRAIN_PY, ICL, "eval", *TEACHERS, "--checkpoint", BEST, "--data-dir", LOCAL_DATA, "--splits", "test_in", "test_q4",
     "--ks", *KS, "--stride", STRIDE, "--goals", EVAL_GOALS, "--starts", EVAL_STARTS,
     "--out", EVAL_DIR / "closed_loop.json"], show=r"^test")

In [ ]:
res = json.loads((EVAL_DIR / "closed_loop.json").read_text())
for split, block in res["splits"].items():
    r = block["results"]
    print(f"\n{split}: success-once / success-at-end, {block['episodes']} episodes per cell")
    print("  teachers (see the goal): " + "   ".join(
        f"{t} {r['teacher_' + t]['success_once']:.0%}/{r['teacher_' + t]['success_at_end']:.0%}" for t in ("ppo", "bc")))
    for t in ("ppo", "bc"):
        line = f"  {t} demos  k=0 none {r[f'{t}_k0_none']['success_once']:.0%}"
        for k in KS[1:]:
            c, w = r[f"{t}_k{k}_correct"], r[f"{t}_k{k}_wrong"]
            line += f" | k={k} correct {c['success_once']:.0%} wrong {w['success_once']:.0%}"
        print(line)
summary = {"session": SESSION, "train_dir": str(TRAIN_DIR), "checkpoint": str(BEST), "data": str(DRIVE_DATA),
           "action_error": json.loads((EVAL_DIR / "action_error.json").read_text()), "closed_loop": res}
(OUTPUT_ROOT / "summary.json").write_text(json.dumps(summary, indent=2))
print("\nSaved", OUTPUT_ROOT / "summary.json")